# 03 - Sequence Generation (how captions become training samples)

The decoder learns to predict a caption **one word at a time**. So a single
caption is turned into several *(input words -> next word)* training pairs.

This notebook explains that idea on one example. The full dataset is **not**
built into memory here - that happens batch-by-batch in **05_data_generator**
(otherwise we would need tens of GB of RAM).

In [1]:
import pickle

import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

In [2]:
with open("../artifacts/tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)
with open("../artifacts/all_captions.pkl", "rb") as f:
    all_captions = pickle.load(f)

In [3]:
vocab_size = len(tokenizer.word_index) + 1
max_length = max(len(caption.split()) for caption in all_captions)

In [5]:
print(f"vocab_size: {vocab_size}")
print(f"max_length: {max_length}")

vocab_size: 8781
max_length: 37


## One caption -> a sequence of integer ids

In [6]:
sample_caption = all_captions[0]

In [7]:
sample_caption

'startseq a child in a pink dress is climbing up a set of stairs in an entry way endseq'

In [8]:
sequence = tokenizer.texts_to_sequences([sample_caption])[0]

In [9]:
sequence

[2, 1, 42, 4, 1, 90, 170, 7, 119, 53, 1, 395, 12, 392, 4, 28, 5201, 693, 3]

In [10]:
for i in range(1, len(sequence)):
    in_seq = sequence[:i]
    out_seq = sequence[i]
    print( f"{in_seq} -> {out_seq}")

[2] -> 1
[2, 1] -> 42
[2, 1, 42] -> 4
[2, 1, 42, 4] -> 1
[2, 1, 42, 4, 1] -> 90
[2, 1, 42, 4, 1, 90] -> 170
[2, 1, 42, 4, 1, 90, 170] -> 7
[2, 1, 42, 4, 1, 90, 170, 7] -> 119
[2, 1, 42, 4, 1, 90, 170, 7, 119] -> 53
[2, 1, 42, 4, 1, 90, 170, 7, 119, 53] -> 1
[2, 1, 42, 4, 1, 90, 170, 7, 119, 53, 1] -> 395
[2, 1, 42, 4, 1, 90, 170, 7, 119, 53, 1, 395] -> 12
[2, 1, 42, 4, 1, 90, 170, 7, 119, 53, 1, 395, 12] -> 392
[2, 1, 42, 4, 1, 90, 170, 7, 119, 53, 1, 395, 12, 392] -> 4
[2, 1, 42, 4, 1, 90, 170, 7, 119, 53, 1, 395, 12, 392, 4] -> 28
[2, 1, 42, 4, 1, 90, 170, 7, 119, 53, 1, 395, 12, 392, 4, 28] -> 5201
[2, 1, 42, 4, 1, 90, 170, 7, 119, 53, 1, 395, 12, 392, 4, 28, 5201] -> 693
[2, 1, 42, 4, 1, 90, 170, 7, 119, 53, 1, 395, 12, 392, 4, 28, 5201, 693] -> 3


In [14]:
sequence

[2, 1, 42, 4, 1, 90, 170, 7, 119, 53, 1, 395, 12, 392, 4, 28, 5201, 693, 3]

In [15]:
padded_sequence = pad_sequences([sequence], maxlen=max_length)[0]

In [16]:
padded_sequence

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    2,    1,   42,    4,
          1,   90,  170,    7,  119,   53,    1,  395,   12,  392,    4,
         28, 5201,  693,    3], dtype=int32)

In [21]:
one_hot = to_categorical(sequence[1], num_classes=vocab_size)

In [22]:
one_hot

array([0., 1., 0., ..., 0., 0., 0.])